# Test vn-address-normalizer

Runs the project from GitHub with a trained tagger:

1. clone + install
2. load the model (attach the output of the training notebook, or a dataset holding an `xlmr-base/` folder, as input)
3. unit tests
4. try your own addresses
5. reproduce the evaluation on the real-style sets (and optionally the synthetic ones)

Run the cells top to bottom. Internet must be on (for `git clone` / `pip`). A GPU is optional and only makes step 5 faster.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/maaL6/vn-address-normalizer.git"
REPO = "/kaggle/working/vn-address-normalizer"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# the code is written for transformers 4.x
import importlib.metadata as md
if int(md.version("transformers").split(".")[0]) >= 5:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40,<5"], check=True)
print("transformers", md.version("transformers"))

sys.path.insert(0, os.path.join(REPO, "src"))  # so this kernel can import vnaddr without a restart

## Load the model

Looks for a folder with `config.json` + `model.safetensors` under `/kaggle/input` (prefers `xlmr-base`) and copies it to `models/tagger/`.

In [ ]:
configs = sorted(glob.glob("/kaggle/input/**/config.json", recursive=True))
model_dirs = [os.path.dirname(c) for c in configs if os.path.exists(os.path.join(os.path.dirname(c), "model.safetensors"))]
print("found:", model_dirs)
preferred = [d for d in model_dirs if d.rstrip("/").endswith("xlmr-base")] or model_dirs
if not preferred:
    raise SystemExit("No model found: add the training notebook output (or a dataset with xlmr-base/) as input.")
shutil.rmtree("models/tagger", ignore_errors=True)
shutil.copytree(preferred[0], "models/tagger")
print("using", preferred[0])

import torch
os.environ["VNADDR_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", os.environ["VNADDR_DEVICE"])

## Unit tests

In [ ]:
!python -m pytest -q

## Command line

In [ ]:
!python -m vnaddr "12 ng trai p.ben thanh q1 tphcm"
!python -m vnaddr "UBND xa Kim Chung, Dong Anh, HN"
!python -m vnaddr "BV đa khoa tỉnh Bình Dương, 5 Phạm Ngọc Thạch, p.Hiệp Thành, TP Thủ Dầu Một, BD"

## Your own addresses

Edit the list and re-run the cell.

In [ ]:
import pandas as pd
from vnaddr.pipeline import Normalizer

n = Normalizer()
my_addresses = [
    "so 1 dai co viet p bach khoa q hai ba trung hn",
    "Q.BT, P.25, SG",
    "Ấp 3 X.Long Phước H.Long Thành Đồng Nai",
    "45 Lê Duẩn, Phường Sài Gòn, TP.HCM",          # already in the new format
    "Tram y te xa Trang Viet, Me Linh, TP Ha Noi",
]
rows = []
for r in n.normalize_batch(my_addresses):
    old, new = r.get("old") or {}, r.get("new") or {}
    rows.append({"input": r["input"], "format": r["fmt"],
                 "old": ", ".join(x for x in (old.get("ward"), old.get("district"), old.get("province")) if x),
                 "new": new.get("full_address"), "split_ward": r["split_ward"], "confidence": r["confidence"]})
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

In [ ]:
# full result for one address
import json
print(json.dumps(n.normalize("Q.BT, P.25, SG"), ensure_ascii=False, indent=2))

## A CSV file

In [ ]:
pd.DataFrame({"id": [1, 2, 3], "address": [
    "12 Nguyễn Trãi, Phường Bến Thành, Quận 1, TP.HCM",
    "thon dong, xa tan thong hoi, huyen cu chi, sg",
    "Số 5 Hùng Vương, P.4, TP Vũng Tàu, BRVT",
]}).to_csv("/kaggle/working/customers.csv", index=False)
!python -m vnaddr --csv /kaggle/working/customers.csv --column address --out /kaggle/working/customers_2025.csv
pd.read_csv("/kaggle/working/customers_2025.csv")

## Reproduce the evaluation

Real-style sets: the model against the `vietnamadminunits` baseline. `final_real` is the set reported in the README.

In [ ]:
!python scripts/evaluate.py --systems baseline rules model --sets data/eval/dev_real.jsonl data/eval/dev2_real.jsonl data/eval/final_real.jsonl --out /kaggle/working/results_real.json

Optional: the synthetic test sets (5,000 + 3,000 addresses). They are regenerated here with the current generator, so the numbers can differ a little from the README. Generating them takes a minute; evaluating the model is quick on a GPU.


In [ ]:
RUN_SYNTHETIC = True
if RUN_SYNTHETIC:
    !python scripts/gen_data.py --train 0
    !python scripts/evaluate.py --systems baseline model --sets data/synth/test.jsonl data/synth/test_hard.jsonl --out /kaggle/working/results_synth.json